# Batch A: 5 Dataset Domination (Colab T4)

**Goal**: Run 4 fusion strategies on 5 BEIR datasets.
Collect official pytrec_eval scores for submission.

**Datasets**: SciFact (5K) + NFCorpus (3.6K) + ArguAna (8.6K) + SCIDOCS (25K) + FiQA (57K)

**Strategies**: Riverbed only | RT full | RRF | Dense only

**Model**: E5-base (110M) — baseline model, larger models in separate run

**Output**: One JSON with all results + agreement stats for Auto design

**Time**: ~2-3 hours on T4

In [1]:
!pip install -q beir sentence-transformers rank-bm25 numpy pytrec-eval-terrier
!nvidia-smi | head -5 || echo 'No GPU'

Sat Mar 28 02:08:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |


In [2]:
import json
import os
import re
import time
from collections import defaultdict

import numpy as np
import pytrec_eval
import torch
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Device: cuda
GPU: Tesla T4


In [3]:
# ── Fusion functions ──

def _norm(results):
    if not results:
        return {}
    vals = [s for _, s in results]
    mn, mx = min(vals), max(vals)
    rng = mx - mn if mx > mn else 1.0
    return {did: (s - mn) / rng for did, s in results}


def simple_rrf(b, d, k=5, bw=1.0, dw=1.2):
    scores = defaultdict(float)
    for rank, (did, _) in enumerate(b):
        scores[did] += bw / (k + rank + 1)
    for rank, (did, _) in enumerate(d):
        scores[did] += dw / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def riverbed_only(b, d, bw=0.8, dw=1.4):
    """Score-Preserving Fusion: normalize + weighted combine."""
    b_n, d_n = _norm(b), _norm(d)
    all_docs = set(b_n) | set(d_n)
    tw = bw + dw
    final = {
        did: (bw * b_n.get(did, 0) + dw * d_n.get(did, 0)) / tw
        for did in all_docs
    }
    return sorted(final.items(), key=lambda x: x[1], reverse=True)


def rt_full(
    b, d, k_low=3, k_high=10, top_n=20,
    boost_max=1.2, score_w=0.5, bw=0.8, dw=1.4,
):
    """Confluence Fusion: adaptive agreement + score preservation."""
    b_set = {did for did, _ in b[:top_n]}
    d_set = {did for did, _ in d[:top_n]}
    union = b_set | d_set
    agreement = len(b_set & d_set) / len(union) if union else 0.0
    tension = 1.0 - agreement
    adaptive_k = max(1, int(k_low + (k_high - k_low) * tension))
    boost = 1.0 + (boost_max - 1.0) * agreement

    rrf_scores = defaultdict(float)
    presence = defaultdict(int)
    for rank, (did, _) in enumerate(b):
        rrf_scores[did] += bw / (adaptive_k + rank + 1)
        presence[did] += 1
    for rank, (did, _) in enumerate(d):
        rrf_scores[did] += dw / (adaptive_k + rank + 1)
        presence[did] += 1
    for did in rrf_scores:
        if presence[did] >= 2:
            rrf_scores[did] *= boost

    b_n, d_n = _norm(b), _norm(d)
    rv = list(rrf_scores.values())
    r_mn, r_mx = min(rv), max(rv)
    r_rng = r_mx - r_mn if r_mx > r_mn else 1.0
    tw = bw + dw

    all_docs = set(rrf_scores) | set(b_n) | set(d_n)
    final = {}
    for did in all_docs:
        r = (rrf_scores.get(did, 0) - r_mn) / r_rng
        s = (bw * b_n.get(did, 0) + dw * d_n.get(did, 0)) / tw
        final[did] = (1 - score_w) * r + score_w * s
    return sorted(final.items(), key=lambda x: x[1], reverse=True)


def measure_agreement(b, d, top_n=20):
    """Measure BM25/Dense agreement for Auto design."""
    b_set = {did for did, _ in b[:top_n]}
    d_set = {did for did, _ in d[:top_n]}
    union = b_set | d_set
    return len(b_set & d_set) / len(union) if union else 0.0


print("Fusion functions ready")

Fusion functions ready


In [4]:
# ── Official evaluation ──

METRICS = {"ndcg_cut_10", "recall_100", "map"}


def evaluate_official(qrels, run_dict):
    """Official BEIR evaluation with pytrec_eval."""
    qrels_int = {
        qid: {did: int(rel) for did, rel in rels.items()}
        for qid, rels in qrels.items()
    }
    evaluator = pytrec_eval.RelevanceEvaluator(qrels_int, METRICS)
    scores = evaluator.evaluate(run_dict)
    result = {}
    for metric in METRICS:
        vals = [scores[qid].get(metric, 0) for qid in scores]
        result[metric] = round(sum(vals) / len(vals), 6)
    return result


print("Evaluation ready (pytrec_eval official)")

Evaluation ready (pytrec_eval official)


In [5]:
# ── Download all datasets ──

BEIR_BASE = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets"

DATASETS = {
    "scifact": f"{BEIR_BASE}/scifact.zip",
    "nfcorpus": f"{BEIR_BASE}/nfcorpus.zip",
    "arguana": f"{BEIR_BASE}/arguana.zip",
    "scidocs": f"{BEIR_BASE}/scidocs.zip",
    "fiqa": f"{BEIR_BASE}/fiqa.zip",
}

BASE_DIR = "datasets"
os.makedirs(BASE_DIR, exist_ok=True)

loaded = {}
for ds_name, url in DATASETS.items():
    data_path = os.path.join(BASE_DIR, ds_name)
    if not os.path.isdir(data_path):
        print(f"Downloading {ds_name}...")
        data_path = util.download_and_unzip(url, BASE_DIR)
    corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")
    loaded[ds_name] = (corpus, queries, qrels)
    print(f"{ds_name}: {len(corpus)} docs, {len(queries)} queries")

print(f"\nAll {len(loaded)} datasets loaded")

  0%|          | 0/5183 [00:00<?, ?it/s]

scifact: 5183 docs, 300 queries


datasets/nfcorpus.zip:   0%|          | 0.00/2.34M [00:00<?, ?iB/s]

  0%|          | 0/3633 [00:00<?, ?it/s]

nfcorpus: 3633 docs, 323 queries


...

  0%|          | 0/25657 [00:00<?, ?it/s]

scidocs: 25657 docs, 1000 queries


datasets/fiqa.zip:   0%|          | 0.00/17.1M [00:00<?, ?iB/s]

  0%|          | 0/57638 [00:00<?, ?it/s]

fiqa: 57638 docs, 648 queries

All 5 datasets loaded


In [6]:
# ── Build BM25 indices ──

def tokenize(text):
    return re.findall(r'\w+', text.lower())


def build_bm25(corpus):
    doc_ids = list(corpus.keys())
    tokenized = [
        tokenize(
            f"{corpus[did].get('title', '')} "
            f"{corpus[did].get('text', '')}"
        )
        for did in doc_ids
    ]
    bm25 = BM25Okapi(tokenized)
    return bm25, doc_ids


def search_bm25(bm25, doc_ids, query, top_k=100):
    scores = bm25.get_scores(tokenize(query))
    top_idx = scores.argsort()[-top_k:][::-1]
    return [
        (doc_ids[i], float(scores[i]))
        for i in top_idx if scores[i] > 0
    ]


bm25_indices = {}
for ds_name, (corpus, _, _) in loaded.items():
    t0 = time.time()
    bm25, doc_ids = build_bm25(corpus)
    bm25_indices[ds_name] = (bm25, doc_ids)
    print(f"{ds_name}: BM25 built in {time.time() - t0:.1f}s")

scifact: BM25 built in 0.7s
nfcorpus: BM25 built in 0.5s
arguana: BM25 built in 0.9s
scidocs: BM25 built in 2.8s
fiqa: BM25 built in 5.6s


In [7]:
# ── Load model ──

MODEL_ID = "intfloat/e5-base-unsupervised"
MODEL_NAME = "E5-base"
PREFIX_Q = "query: "
PREFIX_D = "passage: "

print(f"Loading {MODEL_ID}...")
model = SentenceTransformer(MODEL_ID, device=DEVICE)
print(f"Loaded on {DEVICE}")

Loading intfloat/e5-base-unsupervised...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loaded on cuda


In [8]:
# ── Encode all datasets with checkpoint ──

def encode_corpus(corpus, ds_name):
    tag = MODEL_ID.replace('/', '_').replace('-', '_')
    cache_path = os.path.join(BASE_DIR, f".cache_{ds_name}_{tag}.npz")
    ckpt_path = cache_path + ".ckpt.npz"

    doc_id_list = list(corpus.keys())
    texts = [
        f"{PREFIX_D}{corpus[did].get('title', '')} "
        f"{corpus[did].get('text', '')}".strip()
        for did in doc_id_list
    ]

    if os.path.exists(cache_path):
        data = np.load(cache_path)
        print(f"  Cache hit: {data['embs'].shape}")
        return data["embs"], doc_id_list

    start_idx = 0
    all_embs = []
    if os.path.exists(ckpt_path):
        ckpt = np.load(ckpt_path)
        start_idx = int(ckpt["done"])
        all_embs = [ckpt["embs"]]
        print(f"  Resuming from {start_idx}/{len(texts)}")

    batch_size = 256
    t0 = time.time()
    for i in range(start_idx, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        embs = model.encode(
            batch, normalize_embeddings=True,
            show_progress_bar=False, batch_size=128,
        )
        all_embs.append(embs)
        done = min(i + batch_size, len(texts))
        if done % 5000 < batch_size or done == len(texts):
            partial = np.vstack(all_embs)
            np.savez_compressed(ckpt_path, embs=partial, done=done)
            elapsed = time.time() - t0
            speed = (done - start_idx) / elapsed if elapsed > 0 else 0
            eta = (len(texts) - done) / speed if speed > 0 else 0
            print(f"  {done}/{len(texts)} ({speed:.0f} d/s, ETA {eta:.0f}s)")

    passage_embs = np.vstack(all_embs)
    np.savez_compressed(cache_path, embs=passage_embs)
    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)
    print(f"  Done: {passage_embs.shape} in {time.time() - t0:.1f}s")
    return passage_embs, doc_id_list


embeddings = {}
for ds_name, (corpus, _, _) in loaded.items():
    print(f"\nEncoding {ds_name} ({len(corpus)} docs)...")
    embs, doc_ids = encode_corpus(corpus, ds_name)
    embeddings[ds_name] = (embs, doc_ids)

print("\nAll datasets encoded")


Encoding scifact (5183 docs)...
  5120/5183 (36 d/s, ETA 2s)
  5183/5183 (36 d/s, ETA 0s)
  Done: (5183, 768) in 144.9s

Encoding nfcorpus (3633 docs)...
  3633/3633 (33 d/s, ETA 0s)
  Done: (3633, 768) in 109.6s

Encoding arguana (8674 docs)...
  5120/8674 (45 d/s, ETA 78s)
  8674/8674 (45 d/s, ETA 0s)
  Done: (8674, 768) in 192.0s

Encoding scidocs (25657 docs)...
  5120/25657 (41 d/s, ETA 499s)
  10240/25657 (41 d/s, ETA 381s)
  15104/25657 (41 d/s, ETA 259s)
  20224/25657 (40 d/s, ETA 135s)
  25088/25657 (40 d/s, ETA 14s)
  25657/25657 (40 d/s, ETA 0s)
  Done: (25657, 768) in 643.6s

Encoding fiqa (57638 docs)...
  5120/57638 (48 d/s, ETA 1102s)
  10240/57638 (48 d/s, ETA 984s)
  15104/57638 (48 d/s, ETA 892s)
  20224/57638 (48 d/s, ETA 782s)
  25088/57638 (48 d/s, ETA 682s)
  30208/57638 (47 d/s, ETA 580s)
  35072/57638 (47 d/s, ETA 481s)
  40192/57638 (47 d/s, ETA 374s)
  45056/57638 (46 d/s, ETA 272s)
  50176/57638 (46 d/s, ETA 161s)
  55040/57638 (46 d/s, ETA 56s)
  57638/5763

In [9]:
# ── Run all strategies on all datasets ──

STRATEGIES = {
    "dense_only": lambda b, d: d,
    "rrf": lambda b, d: simple_rrf(b, d),
    "riverbed": lambda b, d: riverbed_only(b, d),
    "rt_full": lambda b, d: rt_full(b, d),
}

ALL_RESULTS = {}
AGREEMENT_STATS = {}

for ds_name, (corpus, queries, qrels) in loaded.items():
    print(f"\n{'=' * 60}")
    print(f"{ds_name.upper()} ({len(corpus)} docs, {len(queries)} queries)")
    print(f"{'=' * 60}")

    bm25, bm25_doc_ids = bm25_indices[ds_name]
    passage_embs, doc_id_list = embeddings[ds_name]

    # Cache BM25 + Dense for all queries
    cached = {}
    agreements = []
    t0 = time.time()

    for qi, (qid, qt) in enumerate(queries.items()):
        bm25_res = search_bm25(bm25, bm25_doc_ids, qt, top_k=100)

        q_emb = model.encode(
            [PREFIX_Q + qt], normalize_embeddings=True,
        )
        sims = (passage_embs @ q_emb.T).flatten()
        idx = np.argsort(sims)[::-1][:100]
        dense_res = [(doc_id_list[i], float(sims[i])) for i in idx]

        cached[qid] = {"bm25": bm25_res, "dense": dense_res}
        agreements.append(measure_agreement(bm25_res, dense_res))

        if (qi + 1) % 200 == 0:
            print(f"  Cached {qi + 1}/{len(queries)} queries")

    cache_time = time.time() - t0
    print(f"  Cached {len(cached)} queries in {cache_time:.1f}s")

    # Agreement stats
    avg_agr = sum(agreements) / len(agreements)
    low_agr = sum(1 for a in agreements if a < 0.3) / len(agreements)
    high_agr = sum(1 for a in agreements if a > 0.5) / len(agreements)
    AGREEMENT_STATS[ds_name] = {
        "mean": round(avg_agr, 4),
        "min": round(min(agreements), 4),
        "max": round(max(agreements), 4),
        "pct_low_tension": round(high_agr * 100, 1),
        "pct_high_tension": round(low_agr * 100, 1),
    }
    print(f"  Agreement: mean={avg_agr:.3f} (low tension {high_agr:.0%} / high tension {low_agr:.0%})")

    # Evaluate each strategy
    ds_results = {}
    for strat_name, strat_fn in STRATEGIES.items():
        run_dict = {}
        for qid in queries:
            e = cached[qid]
            fused = strat_fn(e["bm25"], e["dense"])
            run_dict[qid] = {
                did: float(score)
                for did, score in fused[:100]
            }
        metrics = evaluate_official(qrels, run_dict)
        ds_results[strat_name] = metrics
        n = metrics["ndcg_cut_10"]
        m = metrics["map"]
        print(f"  {strat_name:<15} nDCG@10={n:.4f}  MAP={m:.4f}")

    ALL_RESULTS[ds_name] = ds_results

    # Show winner
    best = max(ds_results.items(), key=lambda x: x[1]["ndcg_cut_10"])
    print(f"  >>> BEST: {best[0]} = {best[1]['ndcg_cut_10']:.4f}")

print("\n" + "=" * 60)
print("ALL DATASETS DONE")
print("=" * 60)


SCIFACT (5183 docs, 300 queries)
  Cached 200/300 queries
  Cached 300 queries in 13.6s
  Agreement: mean=0.203 (low tension 3% / high tension 81%)
  dense_only      nDCG@10=0.7371  MAP=0.6923
  rrf             nDCG@10=0.7503  MAP=0.7057
  riverbed        nDCG@10=0.7576  MAP=0.7175
  rt_full         nDCG@10=0.7557  MAP=0.7141
  >>> BEST: riverbed = 0.7576

NFCORPUS (3633 docs, 323 queries)
  Cached 200/323 queries
  Cached 323 queries in 7.7s
  Agreement: mean=0.159 (low tension 3% / high tension 85%)
  dense_only      nDCG@10=0.3585  MAP=0.1699
  rrf             nDCG@10=0.3609  MAP=0.1751
  riverbed        nDCG@10=0.3633  MAP=0.1748
  rt_full         nDCG@10=0.3666  MAP=0.1773
  >>> BEST: rt_full = 0.3666

ARGUANA (8674 docs, 1406 queries)
  Cached 200/1406 queries
  Cached 400/1406 queries
  Cached 600/1406 queries
  Cached 800/1406 queries
  Cached 1000/1406 queries
  Cached 1200/1406 queries
  Cached 1400/1406 queries
  Cached 1406 queries in 808.2s
  Agreement: mean=0.346 (low te

In [10]:
# ── Summary table ──

print("\n" + "=" * 80)
print("CONFLUENCE FUSION — BEIR Batch A Results (pytrec_eval official)")
print("=" * 80)
print(f"Model: {MODEL_NAME} ({MODEL_ID})")
print(f"Evaluator: pytrec_eval (official BEIR standard)")
print()

# nDCG@10 table
strats = list(STRATEGIES.keys())
header = f"{'Dataset':<12}" + "".join(f"{s:>14}" for s in strats) + f"{'BEST':>10}" + f"{'vs Dense':>10}"
print(header)
print("-" * len(header))

wins = {s: 0 for s in strats}
for ds_name in DATASETS:
    r = ALL_RESULTS[ds_name]
    scores = {s: r[s]["ndcg_cut_10"] for s in strats}
    best_strat = max(scores, key=scores.get)
    best_score = scores[best_strat]
    dense_score = scores["dense_only"]
    wins[best_strat] += 1
    row = f"{ds_name:<12}"
    for s in strats:
        marker = " *" if s == best_strat else "  "
        row += f"{scores[s]:>12.4f}{marker}"
    row += f"{best_score:>10.4f}"
    row += f"{best_score - dense_score:>+10.4f}"
    print(row)

print()
print("Win count:", {s: w for s, w in wins.items() if w > 0})
print("* = best for that dataset")

# Agreement summary
print("\n--- Agreement Stats (for Auto design) ---")
print(f"{'Dataset':<12} {'Mean':>8} {'Low Tension%':>14} {'High Tension%':>15}")
print("-" * 55)
for ds_name in DATASETS:
    a = AGREEMENT_STATS[ds_name]
    print(
        f"{ds_name:<12} {a['mean']:>8.3f}"
        f" {a['pct_low_tension']:>13.1f}%"
        f" {a['pct_high_tension']:>14.1f}%"
    )


CONFLUENCE FUSION — BEIR Batch A Results (pytrec_eval official)
Model: E5-base (intfloat/e5-base-unsupervised)
Evaluator: pytrec_eval (official BEIR standard)

Dataset         dense_only           rrf      riverbed       rt_full      BEST  vs Dense
----------------------------------------------------------------------------------------
scifact           0.7371        0.7503        0.7576 *      0.7557      0.7576   +0.0205
nfcorpus          0.3585        0.3609        0.3633        0.3666 *    0.3666   +0.0081
arguana           0.3174        0.3347 *      0.3286        0.3318      0.3347   +0.0173
scidocs           0.2110        0.2056        0.2116 *      0.2110      0.2116   +0.0006
fiqa              0.4008        0.3962        0.4160 *      0.4122      0.4160   +0.0152

Win count: {'rrf': 1, 'riverbed': 3, 'rt_full': 1}
* = best for that dataset

--- Agreement Stats (for Auto design) ---
Dataset          Mean   Low Tension%   High Tension%
------------------------------------------

In [11]:
# ── Save everything as ONE JSON ──

output = {
    "experiment": "batch_a_beir_domination",
    "model": {"name": MODEL_NAME, "hf_id": MODEL_ID, "params": "110M"},
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU",
    "evaluator": "pytrec_eval 0.5.10 (official BEIR standard)",
    "submission_ready": True,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "strategies": list(STRATEGIES.keys()),
    "datasets": {},
    "agreement_stats": AGREEMENT_STATS,
}

for ds_name in DATASETS:
    corpus, queries, qrels = loaded[ds_name]
    output["datasets"][ds_name] = {
        "corpus_size": len(corpus),
        "num_queries": len(queries),
        "results": ALL_RESULTS[ds_name],
    }

# ONE print, ONE copy
print(json.dumps(output, indent=2))

{
  "experiment": "batch_a_beir_domination",
  "model": {
    "name": "E5-base",
    "hf_id": "intfloat/e5-base-unsupervised",
    "params": "110M"
  },
  "device": "cuda",
  "gpu": "Tesla T4",
  "evaluator": "pytrec_eval 0.5.10 (official BEIR standard)",
  "submission_ready": true,
  "timestamp": "2026-03-28 03:09:27",
  "strategies": [
    "dense_only",
    "rrf",
    "riverbed",
    "rt_full"
  ],
  "datasets": {
    "scifact": {
      "corpus_size": 5183,
      "num_queries": 300,
      "results": {
        "dense_only": {
          "map": 0.692277,
          "recall_100": 0.98,
          "ndcg_cut_10": 0.737066
        },
        "rrf": {
          "map": 0.705718,
          "recall_100": 0.976667,
          "ndcg_cut_10": 0.750332
        },
        "riverbed": {
          "map": 0.717546,
          "recall_100": 0.976667,
          "ndcg_cut_10": 0.757596
        },
        "rt_full": {
          "map": 0.714081,
          "recall_100": 0.976667,
          "ndcg_cut_10": 0.75569